In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# 1. CONFIGURAÇÃO & CAMINHOS
EMB_PATH = Path("../data/processed/08.FRAIL_with_tsvd_10dim.csv")
LLM_PATH = Path("../data/processed/09.llm.csv")
TARGET = "Quedas_ultimos_12meses"

print("--- [PASSO 1] CARREGAR E MERGE ---")

# Carregar
emb = pd.read_csv(EMB_PATH) 
llm = pd.read_csv(LLM_PATH)

# Forçar tipos inteiros nos IDs e Momentos
for k in ["ID_Aluno", "Momento"]:
    emb[k] = pd.to_numeric(emb[k], errors="coerce").astype("Int64")
    llm[k] = pd.to_numeric(llm[k], errors="coerce").astype("Int64")

# Remover duplicados exatos
emb = emb.sort_values(["ID_Aluno","Momento"]).drop_duplicates(subset=["ID_Aluno","Momento"], keep="first")
llm = llm.sort_values(["ID_Aluno","Momento"]).drop_duplicates(subset=["ID_Aluno","Momento"], keep="first")

# Merge
df = emb.merge(llm, on=["ID_Aluno","Momento"], how="left", suffixes=("", "_llmdup"))
df = df.drop(columns=[c for c in df.columns if c.endswith("_llmdup")])

print(f"Shape pós-merge: {df.shape}")

# 2. FILTRAGEM: APENAS IDOSOS COM 4 MOMENTOS (LONGITUDINAL)
print("--- [PASSO 2] FILTRAR APENAS COMPLETOS (MOMENTOS 1, 2, 3, 4) ---")

valid_ids = []
required_moments = {1, 2, 3, 4}

for pid, group in df.groupby("ID_Aluno"):
    moments_present = set(group["Momento"].dropna().unique())
    if moments_present == required_moments:
        valid_ids.append(pid)

# Filtrar o DataFrame
df = df[df["ID_Aluno"].isin(valid_ids)].copy()
df = df.sort_values(by=["ID_Aluno", "Momento"]).reset_index(drop=True)

print(f"Pacientes mantidos (com 4 rastreios): {len(valid_ids)}")
print(f"Shape após filtro longitudinal: {df.shape}")

# 3. CLEANING & FEATURE ENGINEERING
print("--- [PASSO 3] CLEANING & FEATURE ENGINEERING ---")

# A. Corrigir Typos conhecidos
if "HANDGRIP_DIREITA_tentaiva1" in df.columns:
    df = df.rename(columns={"HANDGRIP_DIREITA_tentaiva1": "HANDGRIP_DIREITA_tentativa1"})

# B. Handgrip: Calcular o Máximo por mão e o Melhor Global
hg_right_cols = [c for c in df.columns if re.fullmatch(r"HANDGRIP_DIREITA_tentativa[123]", c)]
hg_left_cols  = [c for c in df.columns if re.fullmatch(r"HANDGRIP_ESQUERDA_tentativa[123]", c)]

if hg_right_cols: df["HANDGRIP_DIREITA_max"] = df[hg_right_cols].astype(float).max(axis=1, skipna=True)
if hg_left_cols:  df["HANDGRIP_ESQUERDA_max"] = df[hg_left_cols].astype(float).max(axis=1, skipna=True)

# Melhor Global (Direita vs Esquerda)
cands = [c for c in ["HANDGRIP_DIREITA_max", "HANDGRIP_ESQUERDA_max"] if c in df.columns]
if cands:
    df["HANDGRIP_BEST"] = df[cands].max(axis=1, skipna=True)

# C. TUG (Timed Up and Go): Priorizar o 'Best' já calculado, ou calcular média
def clean_tug(df, prefix, best_col):
    atts = [c for c in df.columns if c.startswith(prefix) and "tentativa" in c]
    if best_col in df.columns:
        pass 
    elif atts:
        df[f"{prefix}_mean"] = df[atts].astype(float).mean(axis=1, skipna=True)
    return df

df = clean_tug(df, "TIMED_UP_AND_GO_Simples", "Best_TIMED_UP_AND_GO_Simples")
df = clean_tug(df, "TIMED_UP_AND_GO_Dupla_tarefa", "Best_TIMED_UP_AND_GO_Dupla_tarefa")

# 4. REMOVER LEAKAGE & RUÍDO
print("--- [PASSO 4] DROPPING COLUMNS (LEAKAGE & INTERMEDIATES) ---")

drop_list = [
    "Data_avaliacao", "Data_nascimento",
    *hg_right_cols, *hg_left_cols, 
    "HANDGRIP_DIREITA_max", "HANDGRIP_ESQUERDA_max", 
    "HANDGRIP_DIREITA_mean", "HANDGRIP_ESQUERDA_mean",
    *[c for c in df.columns if "TIMED_UP_AND_GO" in c and "tentativa" in c],
    "Quedas_12meses_Hospitalizado_Dias",
    "Quedas_12meses_Situacao",
    "Quedas_12meses_Lesao_Tipo",
    "Quedas_12meses_Lesao_Gravidade",
    "@6_MIN_ANDAR_Desistiu", "Local_de_Avaliacao",
    "Quedas_12meses_Quantas", ""
    "Handrip"
    "Quedas_12meses_Situacao_score","Quedas_12meses_Situacao_score","Age"
]

drop_list += [c for c in df.columns if c.startswith(("Q12M_Sit_Emb_", "Q12M_LesTipo_Emb_", "Q12M_LesLocal_Emb_"))]

df_clean = df.drop(columns=[c for c in drop_list if c in df.columns], errors="ignore")
print(f"Shape final limpo: {df_clean.shape}")

# 5. CARACTERIZAÇÃO DE VARIÁVEIS (NUMÉRICAS, BINÁRIAS E CATEGÓRICAS)
print("--- [PASSO 5] CARACTERIZAÇÃO COMPLETA DE VARIÁVEIS ---")

def categorize_feature(series):
    # 1. Verificar se está vazio
    if series.isnull().all(): return "Empty"
    
    n_unique = series.nunique(dropna=True)
    
    # 2. Binário: Se tiver exatamente 2 valores únicos (ex: 0/1, Sim/Não)
    if n_unique == 2: 
        return "Binary"
    
    # 3. Numérico: Se o tipo de dados for int ou float
    if pd.api.types.is_numeric_dtype(series):
        return "Numeric"
    
    # 4. Categórico vs Texto Livre:
    # Heurística: Se não é número nem binário, e tem poucos valores únicos (<15), é Categórico.
    # Se tiver muitos valores únicos (ex: observações escritas), é Texto/ID.
    if n_unique < 15: 
        return "Categorical"
    
    return "Text/ID"

feat_types = {}
cols_to_check = [c for c in df_clean.columns if c not in ["ID_Aluno", "Momento"]]

for col in cols_to_check:
    feat_types[col] = categorize_feature(df_clean[col])

# Criar DataFrame de referência
df_types = pd.DataFrame(list(feat_types.items()), columns=["Variavel", "Tipo_Inferido"])

# --- IMPRIMIR RESULTADOS ORGANIZADOS ---

print("\n=== RESUMO GERAL ===")
print(df_types["Tipo_Inferido"].value_counts())

print("\n>>> VARIÁVEIS NUMÉRICAS (Para Regressão Linear Simples):")
num_vars = df_types[df_types["Tipo_Inferido"] == "Numeric"]["Variavel"].tolist()
print(num_vars)

print("\n>>> VARIÁVEIS BINÁRIAS (Para Linear Probability Model):")
bin_vars = df_types[df_types["Tipo_Inferido"] == "Binary"]["Variavel"].tolist()
print(bin_vars)

print("\n>>> VARIÁVEIS CATEGÓRICAS (Para Análise de Mudança / One-Hot):")
cat_vars = df_types[df_types["Tipo_Inferido"] == "Categorical"]["Variavel"].tolist()
print(cat_vars)

# Verificar conteúdo das categóricas (para garantir que faz sentido)
if cat_vars:
    print("\n--- Detalhe das Variáveis Categóricas (Valores Únicos) ---")
    for cv in cat_vars:
        unique_vals = df_clean[cv].dropna().unique()
        print(f"{cv}: {unique_vals}")

--- [PASSO 1] CARREGAR E MERGE ---
Shape pós-merge: (6705, 115)
--- [PASSO 2] FILTRAR APENAS COMPLETOS (MOMENTOS 1, 2, 3, 4) ---
Pacientes mantidos (com 4 rastreios): 506
Shape após filtro longitudinal: (2024, 115)
--- [PASSO 3] CLEANING & FEATURE ENGINEERING ---
--- [PASSO 4] DROPPING COLUMNS (LEAKAGE & INTERMEDIATES) ---
Shape final limpo: (2024, 66)
--- [PASSO 5] CARACTERIZAÇÃO COMPLETA DE VARIÁVEIS ---

=== RESUMO GERAL ===
Tipo_Inferido
Numeric    48
Binary     14
Text/ID     2
Name: count, dtype: int64

>>> VARIÁVEIS NUMÉRICAS (Para Regressão Linear Simples):
['Modalidade', 'Freq_semanal', 'ANTROPOMETRIA_Estatura_cm', 'ANTROPOMETRIA_Peso_Kg', 'Escolaridade', 'ATIVIDADE_FISICA_Caminhada_Frequência_Dias_semana', 'ANTROPOMETRIA_Massa_gorda', 'ANTROPOMETRIA_Massa_magra_Kg', 'ANTROPOMETRIA_Gordura_visceral', 'HANDGRIP_DIREITA_tentativa_3', 'Best_TIMED_UP_AND_GO_Simples', 'Percentil_TIMED_UP_AND_GO_Simples', 'Best_TIMED_UP_AND_GO_Dupla_tarefa', 'Percentil_TIMED_UP_AND_GO_Dupla_tarefa',

In [ ]:
from scipy.stats import linregress
import pandas as pd
import numpy as np

# 6. CÁLCULO DE TENDÊNCIAS (Slope e P-value) POR PACIENTE
print("--- [PASSO 6] A CALCULAR TENDÊNCIAS (SLOPE & P-VALUE) ---")

# 1. Recuperar a lista de variáveis numéricas do passo anterior
# (Se não tiveres a variavel 'numeric_vars' guardada, descomenta a linha abaixo e define manualmente)
# numeric_vars = ["HANDGRIP_BEST", "TIMED_UP_AND_GO_Simples_mean", "IMC", "Idade_na_Avaliacao"] 
# Mas idealmente usamos a lista automática:
numeric_vars = df_types[df_types["Tipo_Inferido"] == "Numeric"]["Variavel"].tolist()

# Remover ID e Momento da lista de tendências, pois são as nossas referências
vars_to_test = [v for v in numeric_vars if v not in ["ID_Aluno", "Momento"]]

trend_results = []

# Agrupar por Paciente
# Isto pode demorar alguns segundos dependendo do tamanho da base de dados
total_patients = df_clean["ID_Aluno"].nunique()
count = 0

for pid, group in df_clean.groupby("ID_Aluno"):
    
    # Dicionário para guardar os resultados deste paciente
    p_data = {"ID_Aluno": pid}
    
    # Para cada variável numérica, calcular a regressão
    for var in vars_to_test:
        y = group[var].values # Valores da variável (ex: Força)
        x = group["Momento"].values # Tempo (1, 2, 3, 4)
        
        # Gestão de NaNs: Só podemos calcular se tivermos dados suficientes
        # Cria uma máscara para ignorar momentos onde o valor é nulo
        mask = ~np.isnan(y)
        
        # Precisamos de pelo menos 3 pontos para uma tendência minimamente fiável
        # (A imagem pede os 4 rastreios, mas se falhar 1 valor no meio, calculamos com 3)
        if np.sum(mask) >= 3:
            slope, intercept, r_value, p_value, std_err = linregress(x[mask], y[mask])
            
            p_data[f"{var}_slope"] = slope      # A "Tendência"
            p_data[f"{var}_pvalue"] = p_value   # A "Significância" (Wald Test implícito)
            p_data[f"{var}_r2"] = r_value**2    # Qualidade do ajuste (opcional)
        else:
            # Se não houver dados suficientes, fica NaN
            p_data[f"{var}_slope"] = np.nan
            p_data[f"{var}_pvalue"] = np.nan

    trend_results.append(p_data)
    
    count += 1
    if count % 100 == 0:
        print(f"Processados {count}/{total_patients} pacientes...")

# Transformar em DataFrame
df_trends = pd.DataFrame(trend_results)

print("-" * 40)
print("CÁLCULO TERMINADO")
print(f"Novo DataFrame de Tendências: {df_trends.shape}")
print("-" * 40)

# 7. EXEMPLO DE INTERPRETAÇÃO
# Vamos ver um exemplo para o Handgrip (Força)
if "HANDGRIP_BEST_slope" in df_trends.columns:
    example = df_trends[["ID_Aluno", "HANDGRIP_BEST_slope", "HANDGRIP_BEST_pvalue"]].head(5)
    print("\nEXEMPLO: TENDÊNCIA DE FORÇA (HANDGRIP)")
    print(example)

    print("\n--- GUIA DE INTERPRETAÇÃO ---")
    print("SLOPE (Declive):")
    print(" >  0: O valor está a AUMENTAR ao longo do tempo.")
    print(" <  0: O valor está a DIMINUIR ao longo do tempo.")
    print(" =  0: O valor está ESTÁVEL.")
    print("\nP-VALUE (Significância):")
    print(" < 0.05: A tendência é ESTATISTICAMENTE SIGNIFICATIVA (não é ruído).")
    print(" > 0.05: A tendência pode ser acaso (não há prova forte de mudança).")
# 7. TABELA RESUMO DE TODAS AS VARIÁVEIS NUMÉRICAS
print("\n--- [PASSO 7] GERAR TABELA RESUMO (QUEM MELHOROU/PIOROU?) ---")

summary_list = []

# Iterar sobre as variáveis que testámos (vars_to_test vem do passo anterior)
for var in vars_to_test:
    slope_col = f"{var}_slope"
    pval_col = f"{var}_pvalue"
    
    # Verificação de segurança: se a coluna existe no df_trends
    if slope_col not in df_trends.columns:
        continue
    
    # Extrair séries
    slopes = df_trends[slope_col]
    pvals = df_trends[pval_col]
    
    # Contar Totais Válidos (ignorando quem deu erro ou NaN)
    total_valid = slopes.count()
    
    if total_valid == 0:
        continue

    # 1. Contar Significativos (p_value < 0.05)
    # Aumento Significativo (Slope > 0 e p < 0.05)
    sig_increase = ((slopes > 0) & (pvals < 0.05)).sum()
    
    # Diminuição Significativa (Slope < 0 e p < 0.05)
    sig_decrease = ((slopes < 0) & (pvals < 0.05)).sum()
    
    # Estável / Não Significativo (p >= 0.05)
    stable = ((pvals >= 0.05)).sum()
    
    # 2. Calcular Percentagem de Mudança Real (Impacto na população)
    # Quantos % da população tiveram uma mudança estatisticamente provada?
    pct_changed = ((sig_increase + sig_decrease) / total_valid) * 100
    
    # 3. Tendência Média Global (Média dos slopes de toda a gente)
    mean_slope = slopes.mean()

    # Guardar na lista
    summary_list.append({
        "Variável": var,
        "Total_Avaliados": total_valid,
        "Aumento_Signif (N)": sig_increase,
        "Diminuicao_Signif (N)": sig_decrease,
        "Estavel/Neutro (N)": stable,
        "%_Com_Mudanca_Real": round(pct_changed, 1),
        "Tendencia_Media_Global": round(mean_slope, 4)
    })

# Criar DataFrame
df_summary_num = pd.DataFrame(summary_list)

# Ordenar pelas variáveis onde houve mais agitação (maior % de mudança)
df_summary_num = df_summary_num.sort_values("%_Com_Mudanca_Real", ascending=False).reset_index(drop=True)

# Mostrar Tabela
pd.set_option('display.max_rows', 100) # Garantir que vemos tudo
pd.set_option('display.width', 1000)

print("\nTABELA DE TENDÊNCIAS (Ordenada por % de Pacientes com Mudança Significativa):")
print(df_summary_num)

# Opcional: Guardar em CSV para pores no relatório
# df_summary_num.to_csv("tabela_resumo_tendencias_numericas.csv", index=False)

--- [PASSO 6] A CALCULAR TENDÊNCIAS (SLOPE & P-VALUE) ---
Processados 100/506 pacientes...
Processados 200/506 pacientes...
Processados 300/506 pacientes...
Processados 400/506 pacientes...
Processados 500/506 pacientes...
----------------------------------------
CÁLCULO TERMINADO
Novo DataFrame de Tendências: (506, 145)
----------------------------------------

EXEMPLO: TENDÊNCIA DE FORÇA (HANDGRIP)
   ID_Aluno  HANDGRIP_BEST_slope  HANDGRIP_BEST_pvalue
0        37                -2.31              0.053271
1        42                 1.95              0.469778
2       593                -1.29              0.382025
3       689                -0.26              0.868580
4       745                -1.45              0.377717

--- GUIA DE INTERPRETAÇÃO ---
SLOPE (Declive):
 >  0: O valor está a AUMENTAR ao longo do tempo.
 <  0: O valor está a DIMINUIR ao longo do tempo.
 =  0: O valor está ESTÁVEL.

P-VALUE (Significância):
 < 0.05: A tendência é ESTATISTICAMENTE SIGNIFICATIVA (não é ru

In [ ]:
from scipy.stats import ttest_ind, mannwhitneyu

# ==============================================================================
# 12. CORRELAÇÃO CLÍNICA: QUEM CAI PIORA MAIS RÁPIDO?
# ==============================================================================
print("\n--- [PASSO 9] COMPARAÇÃO: CAIDORES vs NÃO CAIDORES ---")

# 1. Definir o Alvo (Quedas)
target_fall = "Quedas_ultimos_12meses"
# Se já tivermos o padrão calculado no passo anterior, usamos isso
# Se não, calculamos agora quem caiu pelo menos uma vez
if 'df_clean' in locals():
    # Identificar quem caiu pelo menos uma vez nos 4 rastreios
    fallers_ids = df_clean[df_clean[target_fall] == 1]["ID_Aluno"].unique()
else:
    print("Erro: df_clean não encontrado.")
    fallers_ids = []

# 2. Criar Listas de Slopes para Comparar
comparison_results = []

# vars_to_test vem do passo numérico (ex: Handgrip, TUG, Peso)
# Certifica-te que 'df_trends' existe (Passo 6)
if 'df_trends' in locals():
    
    # Criar coluna de grupo no df_trends
    df_trends["Grupo_Queda"] = df_trends["ID_Aluno"].apply(lambda x: "Caidor" if x in fallers_ids else "Nao_Caidor")
    
    for var in vars_to_test:
        slope_col = f"{var}_slope"
        if slope_col not in df_trends.columns: continue
        
        # Separar os grupos
        group_fallers = df_trends[df_trends["Grupo_Queda"] == "Caidor"][slope_col].dropna()
        group_non_fallers = df_trends[df_trends["Grupo_Queda"] == "Nao_Caidor"][slope_col].dropna()
        
        # Só testar se tivermos gente suficiente nos dois lados
        if len(group_fallers) < 5 or len(group_non_fallers) < 5:
            continue
            
        # 3. Teste Estatístico (Mann-Whitney U é mais seguro que T-test para dados não normais)
        stat, p_val = mannwhitneyu(group_fallers, group_non_fallers, alternative='two-sided')
        
        # Médias para interpretação
        mean_fallers = group_fallers.mean()
        mean_non_fallers = group_non_fallers.mean()
        
        # Diferença (Para saber quem está pior)
        # Nota: Depende se "menos" é mau ou bom, mas aqui vemos apenas a diferença matemática
        diff = mean_fallers - mean_non_fallers
        
        comparison_results.append({
            "Variável": var,
            "N_Caidores": len(group_fallers),
            "N_Nao_Caidores": len(group_non_fallers),
            "Slope_Medio_Caidores": round(mean_fallers, 4),
            "Slope_Medio_NaoCaidores": round(mean_non_fallers, 4),
            "Diferenca": round(diff, 4),
            "P_Value_Comparacao": round(p_val, 4)
        })

    # Criar DataFrame
    df_comparison = pd.DataFrame(comparison_results)
    
    # Ordenar por significância (os p-values mais baixos primeiro)
    df_comparison = df_comparison.sort_values("P_Value_Comparacao", ascending=True).reset_index(drop=True)

    # Mostrar Tabela
    print("\nTABELA: O DECLÍNIO É DIFERENTE EM QUEM CAI?")
    # Filtrar apenas as significativas para destaque (ou mostrar top 10)
    print(df_comparison.head(20))
    
else:
    print("Erro: df_trends não encontrado. Corre o PASSO 6 primeiro.")

# DICA DE INTERPRETAÇÃO AUTOMÁTICA
if not df_comparison.empty:
    top_var = df_comparison.iloc[0]
    print(f"\n--- INTERPRETAÇÃO DO TOP 1 ({top_var['Variável']}) ---")
    print(f"Quem cai tem slope médio de {top_var['Slope_Medio_Caidores']}.")
    print(f"Quem NÃO cai tem slope médio de {top_var['Slope_Medio_NaoCaidores']}.")
    
    if top_var['P_Value_Comparacao'] < 0.05:
        print("RESULTADO: A diferença é ESTATISTICAMENTE SIGNIFICATIVA.")
        print("Isto sugere que a velocidade de mudança nesta variável está ligada ao risco de queda.")
    else:
        print("RESULTADO: A diferença NÃO é significativa (p > 0.05).")


--- [PASSO 9] COMPARAÇÃO: CAIDORES vs NÃO CAIDORES ---

TABELA: O DECLÍNIO É DIFERENTE EM QUEM CAI?
                                         Variável  N_Caidores  N_Nao_Caidores  Slope_Medio_Caidores  Slope_Medio_NaoCaidores  Diferenca  P_Value_Comparacao
0                     EQ5D5L_ATIVIDADES_HABITUAIS         242             264                0.0554                   0.0045     0.0508              0.0011
1                       ANTROPOMETRIA_Estatura_cm         242             264               -0.0690                  -0.0424    -0.0266              0.0118
2   Percentil_@6_MIN_ANDAR_Distancia_TOTAL_metros         242             264                0.0050                   0.0705    -0.0655              0.0217
3                           ANTROPOMETRIA_Peso_Kg         242             264               -0.5703                  -0.3860    -0.1843              0.1577
4                            MMSE_COPIA_DESENHO_1         242             264               -0.0194                  -0

In [ ]:
# 8. CÁLCULO DE TENDÊNCIAS PARA VARIÁVEIS BINÁRIAS
print("--- [PASSO 7] A ANALISAR VARIÁVEIS BINÁRIAS (TENDÊNCIAS E PADRÕES) ---")

# 1. Identificar variáveis binárias
binary_vars = df_types[df_types["Tipo_Inferido"] == "Binary"]["Variavel"].tolist()
# Remover ID e Momento se lá estiverem
vars_to_test_bin = [v for v in binary_vars if v not in ["ID_Aluno", "Momento"]]

binary_results = []
count = 0
total_patients = df_clean["ID_Aluno"].nunique()

for pid, group in df_clean.groupby("ID_Aluno"):
    
    p_data = {"ID_Aluno": pid}
    
    for var in vars_to_test_bin:
        y = group[var].dropna().values
        x = group["Momento"].values[:len(y)] # Ajustar x ao tamanho de y (caso haja nans)
        
        # Se tivermos menos de 3 pontos, não calculamos
        if len(y) < 3:
            p_data[f"{var}_slope"] = np.nan
            p_data[f"{var}_pvalue"] = np.nan
            p_data[f"{var}_pattern"] = "Dados Insuficientes"
            continue

        # A. DETEÇÃO DE PADRÃO (Pattern Recognition)
        # Isto é mais útil que o p-value para binários
        soma = np.sum(y)
        if soma == 0:
            pattern = "Sempre Não (0)"  # Ex: Nunca caiu
            slope, p_value = 0.0, 1.0
        elif soma == len(y):
            pattern = "Sempre Sim (1)"  # Ex: Tem sempre depressão
            slope, p_value = 0.0, 1.0
        else:
            # Se variou, calculamos a regressão linear (Linear Probability Model)
            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            
            # Definir o padrão baseada no declive
            if slope > 0:
                pattern = "Tendência de Aumento (Piora)" # 0 -> 1
            elif slope < 0:
                pattern = "Tendência de Descida (Melhora)" # 1 -> 0
            else:
                pattern = "Oscilante/Estável"

        # Guardar resultados
        p_data[f"{var}_slope"] = slope
        p_data[f"{var}_pvalue"] = p_value
        p_data[f"{var}_pattern"] = pattern

    binary_results.append(p_data)
    
    count += 1
    if count % 100 == 0:
        print(f"Processados {count}/{total_patients} binários...")

# Transformar em DataFrame
df_trends_bin = pd.DataFrame(binary_results)

print("-" * 40)
print(f"Análise Binária Terminada. Shape: {df_trends_bin.shape}")

# ==============================================================================
# 9. JUNTAR TUDO (NUMÉRICO + BINÁRIO)
# ==============================================================================
# Agora juntamos as tendências numéricas (feitas antes) com as binárias
if 'df_trends' in locals():
    final_trends = df_trends.merge(df_trends_bin, on="ID_Aluno", how="outer")
else:
    final_trends = df_trends_bin

print(f"DATAFRAME FINAL DE TENDÊNCIAS: {final_trends.shape}")

# Exemplo de Visualização
cols_exemplo = [c for c in final_trends.columns if "Quedas" in c or "Depressao" in c or "FRAGILIDADE" in c]
if cols_exemplo:
    print("\nEXEMPLO (VARIÁVEIS CLÍNICAS):")
    # Mostrar apenas colunas de 'pattern' para ser legível
    pat_cols = [c for c in cols_exemplo if "_pattern" in c]
    print(final_trends[["ID_Aluno"] + pat_cols[:3]].head(10))

# ==============================================================================
# 10. TABELA RESUMO PARA VARIÁVEIS BINÁRIAS
# ==============================================================================
print("\n--- [PASSO 8] GERAR TABELA RESUMO BINÁRIA (QUEM MUDOU DE ESTADO?) ---")

summary_bin_list = []

# Iterar sobre as variáveis binárias testadas
for var in vars_to_test_bin:
    pat_col = f"{var}_pattern"
    slope_col = f"{var}_slope"
    pval_col = f"{var}_pvalue"
    
    # Verificar se as colunas existem
    if pat_col not in df_trends_bin.columns:
        continue
        
    series_pat = df_trends_bin[pat_col]
    total_valid = series_pat.notna().sum()
    
    if total_valid == 0: continue

    # 1. Contagens por Padrão (Baseado no texto que definimos antes)
    # A. Estabilidade
    sempre_nao = (series_pat == "Sempre Não (0)").sum()
    sempre_sim = (series_pat == "Sempre Sim (1)").sum()
    
    # B. Mudança
    piora = (series_pat == "Tendência de Aumento (Piora)").sum()   # Slope > 0
    melhora = (series_pat == "Tendência de Descida (Melhora)").sum() # Slope < 0
    
    # C. Oscilante (Sobe e desce sem tendência clara)
    oscilante = (series_pat == "Oscilante/Estável").sum()

    # 2. Significância Estatística
    # Quantos tiveram p < 0.05 E tiveram alguma mudança (slope != 0)
    # (Ignoramos os "Sempre..." porque o p-value deles é 1.0)
    sig_change = ((df_trends_bin[pval_col] < 0.05) & (df_trends_bin[slope_col] != 0)).sum()

    # 3. Métricas Derivadas
    # % de Prevalência (Quem teve o problema pelo menos uma vez?)
    # = Toda a gente MENOS quem teve "Sempre Não"
    prevalencia_total = total_valid - sempre_nao
    pct_afetados = (prevalencia_total / total_valid) * 100
    
    summary_bin_list.append({
        "Variável": var,
        "Total_Avaliados": total_valid,
        "Sempre_Não (Saudável)": sempre_nao,
        "Sempre_Sim (Crónico)": sempre_sim,
        "Piora (Novo Caso)": piora,
        "Melhora (Recuperação)": melhora,
        "Mudança_Significativa (N)": sig_change,
        "%_População_Afetada": round(pct_afetados, 1)
    })

# Criar DataFrame
df_summary_bin = pd.DataFrame(summary_bin_list)

# Ordenar: Mostrar primeiro as doenças/problemas mais comuns (% Afetada)
# Ou podes ordenar por "Piora" para ver onde estão a surgir novos problemas
df_summary_bin = df_summary_bin.sort_values("%_População_Afetada", ascending=False).reset_index(drop=True)

# Mostrar Tabela
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

print("\nTABELA DE TENDÊNCIAS BINÁRIAS (Ordenada por Prevalência do Problema):")
print(df_summary_bin)

# Opcional: Salvar
# df_summary_bin.to_csv("tabela_resumo_binarias.csv", index=False)

--- [PASSO 7] A ANALISAR VARIÁVEIS BINÁRIAS (TENDÊNCIAS E PADRÕES) ---
Processados 100/506 binários...
Processados 200/506 binários...
Processados 300/506 binários...
Processados 400/506 binários...
Processados 500/506 binários...
----------------------------------------
Análise Binária Terminada. Shape: (506, 43)
DATAFRAME FINAL DE TENDÊNCIAS: (506, 188)

EXEMPLO (VARIÁVEIS CLÍNICAS):
   ID_Aluno  Quedas_ultimos_12meses_pattern Quedas_12meses_Lesao_Sim_Nao_pattern Quedas_12meses_Assistencia_hospitalar_Sim_Não_pattern
0        37                  Sempre Não (0)                       Sempre Não (0)                                     Sempre Não (0)   
1        42                  Sempre Não (0)                       Sempre Não (0)                                     Sempre Não (0)   
2       593                  Sempre Não (0)                       Sempre Não (0)                                     Sempre Não (0)   
3       689    Tendência de Aumento (Piora)       Tendência de Descida 

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

print("\n--- [PASSO 6] TESTE ESTATÍSTICO: CHI-QUADRADO (BINÁRIO vs TARGET) ---")

# 1. Define lists
results = []
# Ensure we don't test the target against itself
features_to_test = [v for v in bin_vars if v != TARGET]

print(f"Target: {TARGET}")
print(f"Testing {len(features_to_test)} binary features...")

# 2. Loop through the binary variables identified in your Step 5
for feature in features_to_test:
    # Create the Contingency Table (Crosstab)
    # This counts frequencies: e.g., [Did Not Fall + No Diabetes], [Fell + Has Diabetes], etc.
    contingency_table = pd.crosstab(df_clean[feature], df_clean[TARGET])
    
    if contingency_table.shape == (2, 2):
        try:
            # Run Chi-Square
            stat, pvalue, dof, expected = chi2_contingency(contingency_table)
            
            results.append({
                'Feature': feature,
                'P-Value': pvalue,
                'Significant': pvalue < 0.05, # Using standard alpha of 0.05
                'Chi2_Stat': stat
            })
        except Exception as e:
            print(f"Could not calculate for {feature}: {e}")
    else:
        # Handle cases where a variable might be constant (e.g., everyone has 0)
        print(f"Skipping {feature}: Not a 2x2 table (Shape: {contingency_table.shape})")

# 3. Create a clean Results DataFrame
stats_df = pd.DataFrame(results)

if not stats_df.empty:
    # Sort by significance (lowest p-value on top)
    stats_df = stats_df.sort_values(by='P-Value', ascending=True)
    
    print("\n>>> RESULTADOS SIGNIFICATIVOS (p < 0.05):")
    print(stats_df[stats_df['Significant'] == True][['Feature', 'P-Value', 'Chi2_Stat']])
    
    print("\n>>> TODOS OS RESULTADOS:")
    print(stats_df)
else:
    print("Nenhuma correlação calculável encontrada.")


--- [PASSO 6] TESTE ESTATÍSTICO: CHI-QUADRADO (BINÁRIO vs TARGET) ---
Target: Quedas_ultimos_12meses
Testing 13 binary features...

>>> RESULTADOS SIGNIFICATIVOS (p < 0.05):
                                         Feature        P-Value    Chi2_Stat
1                   Quedas_12meses_Lesao_Sim_Nao  1.858013e-220  1004.532579
2  Quedas_12meses_Assistencia_hospitalar_Sim_Não   4.694273e-82   368.165757
4               Quedas_12meses_Medo_cair_Sim_Não   3.683170e-21    89.137424
0                                         Genero   1.325739e-17    72.955982
3           Quedas_12meses_Hospitalizado_Sim_Não   5.222502e-13    52.119452
5                 ATIVIDADE_FISICA_Geral_Sim_Não   3.299806e-03     8.633811
9        TIMED_UP_AND_GO_Dupla_tarefa_Apoio_maos   4.008994e-03     8.279737
7             TIMED_UP_AND_GO_Simples_Apoio_mãos   3.695441e-02     4.352511

>>> TODOS OS RESULTADOS:
                                          Feature        P-Value  Significant    Chi2_Stat
1              